# Palaeogeography

## Loading packages

In [1]:
library("palaeoverse")
library("divvy")
library("lwgeom")
library("h3jsr")
library("ggplot2")
library("ggpubr")
library("sf")
library("raster")
library("geojsonio")
library("sp")
library("dplyr") 

Linking to liblwgeom 3.0.0beta1 r16016, GEOS 3.8.0, PROJ 6.3.1

Linking to GEOS 3.8.0, GDAL 3.0.4, PROJ 6.3.1; sf_use_s2() is TRUE


Attachement du package : ‘sf’


L'objet suivant est masqué depuis ‘package:lwgeom’:

    st_perimeter


Le chargement a nécessité le package : sp

Registered S3 method overwritten by 'geojson':
  method        from     
  print.geojson geojsonsf


Attachement du package : ‘geojsonio’


L'objet suivant est masqué depuis ‘package:base’:

    pretty



Attachement du package : ‘dplyr’


Les objets suivants sont masqués depuis ‘package:raster’:

    intersect, select, union


Les objets suivants sont masqués depuis ‘package:stats’:

    filter, lag


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, setdiff, setequal, union




## Setting up environement

In [2]:
sf_use_s2(FALSE)

Spherical geometry (s2) switched off



## Creating function to aproximate point as squares

In [3]:
make_square <- function(data){
    return(st_polygon(list(cbind(c(data[1] - 0.51, data[1] + 0.51, data[1] + 0.51, data[1] - 0.51, data[1] - 0.51),
                c(data[2] - 0.51, data[2] - 0.51, data[2] + 0.51, data[2] + 0.51, data[2] - 0.51)))))
}

## Loading occurence data

### Extinct taxa

In [4]:
table_coordinate_extinct <- read.table("Occurences_extinct_Orectolobiformes.tsv", header = TRUE, sep ="\t")

In [5]:
table_coordinate_extinct <- cbind(table_coordinate_extinct[, c(4)], apply(table_coordinate_extinct[,c(6,7)], 1, mean), table_coordinate_extinct[, c(10, 11)])

In [6]:
colnames(table_coordinate_extinct) <- c("scientificName", "age", colnames(table_coordinate_extinct)[c(3,4)])

### Extant taxa

In [7]:
table_coordinate_extant <- read.table("Occurences_extant_Orectolobiformes.tsv", header = TRUE, sep ="\t")

## Paleocoordinate

In [8]:
paleo_coordinate <- palaeorotate(table_coordinate_extinct, lng = "Longitude", lat = "Latitude", age = "age", model = c("MERDITH2021", "PALEOMAP", "MATTHEWS2016_pmag_ref"))

MERDITH2021

PALEOMAP

MATTHEWS2016_pmag_ref

Warning message in palaeorotate(table_coordinate_extinct, lng = "Longitude", lat = "Latitude", :
“Palaeocoordinates could not be reconstructed for all points.
Either assigned plate does not exist at time of reconstruction or the plate rotation model(s) does not cover the age of reconstruction.”


In [9]:
paleo_coordinate_cleaned <- na.omit(paleo_coordinate)

In [10]:
data_epochs <- data.frame(Epoch = c("Today_Miocene", "Oligocene", "Eocene", "Palaeocene", "Campanian_Maastrichian", "Cenomanian_Santonian", "Early Cretaceous", "Jurassic"),
        Age_min = c(0, 23.031, 33.901, 56.001, 66.001, 86.3, 100.501, 145.001), 
        Age_max = c(23.03, 33.9, 56, 66, 86.3, 100.5, 145, 201.4), 
        Age = c(mean(0, 23.03), mean(23.03, 33.9), mean(33.9, 56), mean(56, 66), mean(66, 86.3), mean(86.3, 100.5), mean(100.5, 145), mean(145,201.4)))

In [11]:
# Set up bounding box
ras <- raster::raster(res = 5, val = 1)
ras <- rasterToPolygons(x = ras, dissolve = TRUE)

# Robinson projection
bb <- sf::st_as_sf(x = ras)
bb <- st_transform(x = bb, crs = sf::st_crs(4326))
bbox <- st_graticule(crs = st_crs("ESRI:54030"), lat = c(-89.9, 89.9), lon = c(-179.9, 179.9))

In [ ]:
for(i in 1:nrow(data_epochs)){
    temp_url <- paste("https://gws.gplates.org/reconstruct/coastlines/?&time=", data_epochs$Age[i], "&model=MERDITH2021&anchor_plate_id=1", sep = "")
    temp_name <- paste("Plot/Merdith_2021/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    gpm <- read_sf(temp_url)
    gpm <- gpm  %>% st_union()
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_MERDITH2021, latitude = temp_paleo_data$p_lat_MERDITH2021)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinates <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinates)
}

although coordinates are longitude/latitude, st_union assumes that they are
planar



In [ ]:
for(i in 1:nrow(data_epochs)){
    temp_url <- paste("https://gws.gplates.org/reconstruct/coastlines/?&time=", data_epochs$Age[i], "&model=PALEOMAP&anchor_plate_id=1", sep = "")
    temp_name <- paste("Plot/PALEOMAP/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    gpm <- read_sf(temp_url)
    gpm <- gpm  %>% st_union()
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinate <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

In [ ]:
for(i in 1:nrow(data_epochs)){
    temp_url <- paste("https://gws.gplates.org/reconstruct/coastlines/?&time=", data_epochs$Age[i], "&model=MATTHEWS2016&anchor_plate_id=1", sep = "")
    temp_name <- paste("Plot/Matthews_2016/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    gpm <- read_sf(temp_url)
    gpm <- gpm  %>% st_union()
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_MATTHEWS2016, latitude = temp_paleo_data$p_lat_MATTHEWS2016)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    palaeo_coordinates <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = coords_sf, aes(color = "#FAC898")) +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinates)
}

In [ ]:
data_epochs <- data.frame(Epoch = c("10", "20", "30", "40", "50", "60", "70", "80", "90", "100", "110", "120", "130", "140", "150", "160"),
        Age_min = c(0, 10.001, 20.001, 30.001, 40.001, 50.001, 60.001, 70.001, 80.001, 90.001, 100.001, 110.001, 120.001, 130.001, 140.001, 150.001), 
        Age_max = c(10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160), 
        Age = c(5, 15, 25, 35, 45, 55, 65, 75, 85, 95, 105, 115, 125, 135, 145, 155))

In [ ]:
for(i in 1:nrow(data_epochs)){
    temp_name <- paste("Plot/PALEOMAP/paleocoordinates", data_epochs$Epoch[i], ".pdf", sep ="")
    temp_Map <- paste("PaleoDEMS/", data_epochs$Age[i], "Ma.csv", sep ="")
    
    temp_paleo_data <- paleo_coordinate_cleaned[paleo_coordinate_cleaned$age > data_epochs$Age_min[i] & paleo_coordinate_cleaned$age < data_epochs$Age_max[i], ]
    pal_cor <- data.frame(longitude = temp_paleo_data$p_lng_PALEOMAP, latitude = temp_paleo_data$p_lat_PALEOMAP)
    
    coordinates(pal_cor) <- ~longitude + latitude
    proj4string(pal_cor) <- CRS("+proj=longlat +datum=WGS84")
    SpatialPoints(pal_cor, proj4string = CRS("ESRI:54030"))
    coords_robinson <- spTransform(pal_cor, CRS("+proj=robin +datum=WGS84"))
    coords_sf <- st_as_sf(coords_robinson)
    
    tab <- read.table(temp_Map, sep = ",")
    temp_tab_0 <- tab
    temp_tab_0$V1[temp_tab_0$V1 >= 180 ] <- 179.48
    temp_tab_0$V1[temp_tab_0$V1 <= -180] <- -179.48
    temp_tab_0$V2[temp_tab_0$V2 >= 90 ] <- 89.48
    temp_tab_0$V2[temp_tab_0$V2 <= -90] <- -89.48
    temp_tab_01 <- temp_tab_0[temp_tab_0$V3 >= -200 & temp_tab_0$V3 <= 0,]
    temp_tab_02 <- temp_tab_0[temp_tab_0$V3 > 0,]
    temp_tab_01 <- temp_tab_01[,-3]
    temp_tab_02 <- temp_tab_02[,-3]

    
    data_inner_sea <- st_sfc(apply(temp_tab_01, MARGIN = 1,  FUN = make_square))
    data_inner_sea <- data_inner_sea %>% st_union()
    sf::st_crs(data_inner_sea) <- sf::st_crs("+proj=longlat +datum=WGS84") 

    data_land <- st_sfc(apply(temp_tab_02, MARGIN = 1,  FUN = make_square))
    data_land <- data_land %>% st_union()
    sf::st_crs(data_land) <- sf::st_crs("+proj=longlat +datum=WGS84") 
    
    palaeo_coordinate <- ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = data_land, fill = "#BEB4B0", colour = NA) +
        geom_sf(data = bbox) +
        geom_sf(data = data_inner_sea, fill =  "#81CDC6", colour = NA) + 
        geom_sf(data = coords_sf, colour = "#FAC898") +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()
    
    ggsave(filename = temp_name, plot = palaeo_coordinate)
}

## Extant occurences

In [ ]:
table_coordinate_extant_without_whale_shark <- table_coordinate_extant[!table_coordinate_extant$Species == "Rhincodon typus", ]

In [ ]:
table_coordinate_extant_without_whale_shark <- table_coordinate_extant[table_coordinate_extant_without_whale_shark$References == "Inaturalist", ]

In [ ]:
table_coordinate_extant_subsampled <- table_coordinate_extant_without_whale_shark %>% 
      group_by(Species) %>%
          slice(sample(min(10, n())))

In [ ]:
unique_sp <- unique(table_coordinate_extant_without_whale_shark$Species)

for(i in 1:length(unique_sp)){
    
    temp <- table_coordinate_extant_without_whale_shark[table_coordinate_extant_without_whale_shark$Species == unique_sp[i], ]

    
    if(i == 1){
        if(nrow(temp) < 5){
            table_coordinate_extant_subsampled <- temp[,c(4,5)]
        }
        
        else{
            table_coordinate_extant_subsampled <- clustr(dat = temp, xy = c("Longitude", "Latitude"), iter = 1, nSite = 5, distMax = 1000)[[1]]
        }

    }
    
    else{
        
        if(nrow(temp) < 5){
            table_coordinate_extant_subsampled <- rbind(table_coordinate_extant_subsampled,temp[,c(4,5)])
        }
        else{
            table_coordinate_extant_subsampled <- rbind(table_coordinate_extant_subsampled, clustr(dat = temp, xy = c("Longitude", "Latitude"), iter = 1, nSite = 5, distMax = 1000)[[1]])
        }
    }
    
}

In [ ]:
gpm <- read_sf("https://gws.gplates.org/reconstruct/coastlines/?&time=1&model=PALEOMAP&anchor_plate_id=1")
gpm <- gpm  %>% st_union()
    
cor_extinct <- data.frame(longitude = paleo_coordinate_cleaned$Longitude , latitude = paleo_coordinate_cleaned$Latitude)
cor_extant <- data.frame(longitude = table_coordinate_extant_subsampled$Longitude , latitude = table_coordinate_extant_subsampled$Latitude)

coordinates(cor_extinct) <- ~longitude + latitude
proj4string(cor_extinct) <- CRS("+proj=longlat +datum=WGS84")
SpatialPoints(cor_extinct, proj4string = CRS("ESRI:54030"))
coords_robinson_extinct <- spTransform(cor_extinct, CRS("+proj=robin +datum=WGS84"))
cor_extinct_sf <- st_as_sf(coords_robinson_extinct)

coordinates(cor_extant) <- ~longitude + latitude
proj4string(cor_extant) <- CRS("+proj=longlat +datum=WGS84")
SpatialPoints(cor_extant, proj4string = CRS("ESRI:54030"))
coords_robinson_extant <- spTransform(cor_extant, CRS("+proj=robin +datum=WGS84"))
cor_extant_sf <- st_as_sf(coords_robinson_extant)

palaeo_coordinate <- gpm  %>% ggplot() +
        geom_sf(data = bb, fill = "#ADD8E6", colour = NA) +
        geom_sf(data = gpm, fill = "#BEB4B0", colour = "black", alpha = 1) +
        geom_sf(data = bbox) +
        geom_sf(data = cor_extant_sf, colour = "#CB99CC") +
        geom_sf(data = cor_extinct_sf, colour = "#FAC898") +
        coord_sf(crs = sf::st_crs("ESRI:54030")) +
        theme_void()

ggsave(filename = "Extant_distribution_Fossil_site.pdf", plot = palaeo_coordinate)

In [ ]:
plot(x=paleo_coordinate_cleaned$age, y=paleo_coordinate_cleaned$p_lat_PALEOMAP, axes = FALSE, xlab = NA, ylab = "Depth (m)")
lines(lowess(x=paleo_coordinate_cleaned$age, y=paleo_coordinate_cleaned$p_lat_PALEOMAP), col = 3, lwd = 3)
box()
axis(2)
axis_geo(side = 1, intervals = list("epoch", "period"),
    abbr = list(TRUE, FALSE))